In [1]:
import os
import cv2
from tqdm import tqdm

In [ ]:
# =====================================================
# DIRECTORIOS
# =====================================================

INPUT_DIR  = "dataset/Training_LR_escala2"
OUTPUT_DIR = "dataset/CLAHE/Training_CLAHE_escala2"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# =====================================================
# CLAHE CON MÁSCARA
# =====================================================

def clahe_con_mascara(imagen_gray,
                      clip_limit=2.0,
                      tile_grid_size=(8, 8)):

    _, mascara_rostro = cv2.threshold(
        imagen_gray,
        15,
        255,
        cv2.THRESH_BINARY
    )

    mascara_suave = cv2.GaussianBlur(
        mascara_rostro,
        (5, 5),
        0
    )

    mascara_norm = mascara_suave.astype(np.float32) / 255.0

    img_filtrada = cv2.bilateralFilter(
        imagen_gray,
        d=3,
        sigmaColor=20,
        sigmaSpace=20
    )

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )

    img_clahe = clahe.apply(img_filtrada)

    resultado = (
        img_clahe.astype(np.float32) * mascara_norm +
        imagen_gray.astype(np.float32) * (1.0 - mascara_norm)
    )

    return resultado.astype(np.uint8)

In [4]:
# =====================================================
# BUSCAR IMÁGENES
# =====================================================

imagenes = []

for root, dirs, files in os.walk(INPUT_DIR):

    for file in files:

        if file.lower().endswith(".png"):

            imagenes.append(
                os.path.join(root, file)
            )

print(f"Imágenes encontradas: {len(imagenes)}")

Imágenes encontradas: 360


In [ ]:
# =====================================================
# GENERAR DATASET CLAHE en escala x4
# =====================================================

for ruta_img in tqdm(imagenes):

    img = cv2.imread(
        ruta_img,
        cv2.IMREAD_GRAYSCALE
    )

    if img is None:
        continue

    resultado = clahe_con_mascara(img)

    # Mantener estructura s1, s2, ..., s40

    relative_path = os.path.relpath(
        ruta_img,
        INPUT_DIR
    )

    save_path = os.path.join(
        OUTPUT_DIR,
        relative_path
    )

    os.makedirs(
        os.path.dirname(save_path),
        exist_ok=True
    )

    cv2.imwrite(
        save_path,
        resultado
    )

print("\nDataset CLAHE escala x2 generado correctamente.")
print(f"Total imágenes: {len(imagenes)}")
print(f"Guardado en: {OUTPUT_DIR}")

100%|██████████| 360/360 [00:12<00:00, 28.98it/s]


Dataset CLAHE escala x4 generado correctamente.
Total imágenes: 360
Guardado en: dataset/CLAHE/Training_CLAHE_escala4
